# Pythonによるデータ解析演習｜最終レポート

**課題：UCIのデータを1つ選び、Pythonで探索・分析して、わかったことを説明する。**

- 学籍番号・氏名：【記入】
- レポートタイトル：【記入】

**目標**：Pythonや生成AIなどを道具として使い、分析する問いを設定し、結果を自分で解釈・考察する。

## 課題の条件と評価

1. [UCI Machine Learning Repository](https://archive.ics.uci.edu/datasets) から**表形式のデータを1つ**選ぶ。
2. データの内容を確認し、**目的に沿った図・表を2つ以上**示して説明する。
3. **多変量解析手法を1つ以上**適用し、選択理由と結果を示す。
4. **考察を重視**する。何がわかったかだけでなく、結果の意味と解釈上の注意点を書く。
5. 実行できるNotebook（`.ipynb`）を提出する。

| 評価項目 | 配点 |
|:--|--:|
| データ選択と問い | 15 |
| データ理解・探索的解析 | 25 |
| 多変量解析の選択・実施 | 20 |
| **考察・結論** | **35** |
| コードの再現性 | 5 |

**生成AIの利用は自由**です。利用の有無やコードの長さではなく、問いに対して分析を選び、結果を理解して説明できているかを評価します。主な多変量解析は1手法で十分です。

## 1. データの選択

- **データ名・UCI URL**：【記入】
- **選んだ理由**：【記入】
- **1行が表すもの・注目する変数**：【記入】
- **調べたいこと**：【記入。EDA後に変更してよい】

※ UCIが設定した予測課題をそのまま解く必要はありません。

## 2. データの取得

次のセルの `DATASET_ID` をUCIページに記載された整数IDに変更してください（例：Wine Quality は `186`）。最初の実行にはインターネット接続が必要です。

In [ ]:
%pip -q install ucimlrepo

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ucimlrepo import fetch_ucirepo

DATASET_ID = None  # 【変更】例：186

if DATASET_ID is None:
    print('DATASET_ID を指定してください。')
    dataset = None
    df = pd.DataFrame()
else:
    dataset = fetch_ucirepo(id=DATASET_ID)
    if dataset.data.original is None:
        raise ValueError('データを読み込めませんでした。別のデータを選ぶかUCIの説明を確認してください。')
    df = dataset.data.original.copy()
    print('データ名:', dataset.metadata.name)
    print('観測数・列数:', df.shape)
    display(df.head())

## 3. データを理解する

変数の意味・単位、欠損値、値の分布を確認します。必要に応じて出典ページの変数説明も読んでください。

In [ ]:
if dataset is None:
    print('データを取得してから実行してください。')
else:
    print('欠損数・データ型')
    display(pd.DataFrame({'データ型': df.dtypes, '欠損数': df.isna().sum()}))
    print('数値変数の記述統計')
    display(df.describe(include='number').T)
    if dataset.variables is not None:
        print('UCIの変数説明（冒頭）')
        display(dataset.variables.head(20))

**【記入】** データの規模、重要な変数の意味、欠損値など注意すべき点を説明してください。表をそのまま貼るだけで終わらないこと。

## 4. 探索的データ解析（EDA）

関心のある変数について**図・表を2つ以上**作成してください。まずは下の例を参考に、変数間の関係や分布を調べてみましょう。

In [ ]:
# 動作例：数値列があれば、最初の1列でヒストグラムを描く
if dataset is not None:
    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) > 0:
        col = numeric_cols[0]
        df[col].dropna().hist(bins=25)
        plt.xlabel(col)
        plt.ylabel('Count')
        plt.show()
# 提出用の図は、調べたい内容に合わせて選び直してください。

In [ ]:
# 【記入】ここに自分のEDAコード（図・表を2つ以上）を書く

**【記入】** 各図・表から読み取れる特徴と、そこから生じた疑問を書いてください。

**EDAを踏まえた分析の問い**：【記入】

## 5. 多変量解析の方法を選ぶ

- **適用する手法と、その理由**：【記入】
- **使用する変数・前処理（必要な場合）**：【記入】

| 明らかにしたいこと | 手法の例 |
|:--|:--|
| 複数の変数を少数の軸で要約 | 主成分分析 |
| 複数の変数と目的変数の関係 | 重回帰分析 |
| 似た観測をまとめる | クラスター分析 |

**注意**：標準化や欠損値の除外などは、機械的に行わず必要性を考えましょう。

<details>
<summary>参考：Pythonコードの最小例（必要な人だけ開く）</summary>

**主成分分析**
```python
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
cols = ['列A', '列B', '列C']  # 自分の数値列に変更
x = analysis_df[cols].dropna()
z = StandardScaler().fit_transform(x)
model = PCA(n_components=2).fit(z)
print(model.explained_variance_ratio_)
print(pd.DataFrame(model.components_.T, index=cols))
```

**重回帰分析**
```python
from sklearn.linear_model import LinearRegression
cols = ['説明変数A', '説明変数B']
y_col = '目的変数'
s = analysis_df[cols + [y_col]].dropna()
model = LinearRegression().fit(s[cols], s[y_col])
print(pd.Series(model.coef_, index=cols))
print('学習データでの決定係数:', model.score(s[cols], s[y_col]))
```

**クラスター分析**
```python
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
cols = ['列A', '列B', '列C']
z = StandardScaler().fit_transform(analysis_df[cols].dropna())
labels = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(z)
print(pd.Series(labels).value_counts())
```

いずれも変数名・前処理を自分のデータに合わせて変更してください。特に標準化の要否、観測数、結果の意味を確認しましょう。

</details>

In [ ]:
# 【必要に応じて修正】分析用のデータを作る
analysis_df = df.copy()
# 例：不要な列の除外、欠損値の処理、カテゴリ変数の変換など

## 6. 多変量解析の実施と結果

選んだ手法を実行し、**分析に用いた変数・観測数**と主な結果を図・表・数値で示してください。

In [ ]:
# 【記入】多変量解析のコードを書く

**【記入】** 結果として観察された事実を述べてください（例：どの変数と何が関連したか、主成分がどんな特徴を示したか）。**結果の意味づけは次の考察で行います。**

## 7. 考察・結論【重視】

**図や計算結果の説明を繰り返すだけでなく、得られた知見について自分の言葉で考察してください。** 次の内容を文章でまとめましょう。

1. **わかったこと**：最初の問いにどう答えられるか。第4・6節の結果を根拠として挙げる。
2. **結果の意味**：なぜそのような傾向が見られると考えるか。予想との違いや、新たな気づきはあるか。
3. **解釈上の注意点**：どこまで結論を言えるか。データの偏り、変数の尺度、解析手法の前提、因果と相関の違いなどから重要な点を述べる。
4. **結論・今後の課題**：今回の分析を短く総括し、さらに調べたいことがあれば述べる。

※ **別の条件で分析し直す感度分析は必須ではありません。** ただし、例えば回帰分析の残差に目立つ傾向がないか、PCAで変数の尺度をどう扱ったか、といった基本的な確認は必要です。相関から因果関係を断定しないことにも注意しましょう。

**【考察・結論をここに記述】**

### 出典・参考資料
- UCIデータページのURL：【記入】
- 使用した資料（原著論文、講義資料、公式ドキュメント等）：【記入】

※ 必要な引用表記・利用条件はUCIの各データページで確認してください。

---
**提出前チェック**

- [ ] 図・表を2つ以上作り、その意味を書いた。
- [ ] 多変量解析の選択理由と結果を書いた。
- [ ] 問いに答える**考察**と、解釈の限界を書いた。
- [ ] データの出典を明記した。
- [ ] `DATASET_ID` と【記入】を埋め、**ランタイム再起動 → すべて実行**でエラーがないことを確認した。